# Module 1: Data Preparation and Locked Baseline

This notebook handles acquiring the data, cleaning, feature engineering to the locked 8-feature schema, and training baseline models.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Acquire and freeze raw dataset

In [2]:
# Fetch dataset
dataset = fetch_ucirepo(id=697)
X = dataset.data.features
y = dataset.data.targets
raw_df = X.join(y)

# Save raw dataset
raw_df.to_csv('../data/raw/student_dropout_raw.csv', index=False)
print("Dataset shape:", raw_df.shape)

Dataset shape: (4424, 37)


## 2. Target Transformation

In [3]:
target_map = {
    'Dropout': 1,
    'Enrolled': 0,
    'Graduate': 0
}
df = raw_df.copy()
df['dropout'] = df['Target'].map(target_map).astype('int8')
print(df['dropout'].value_counts())
print(df['dropout'].value_counts(normalize=True).mul(100).round(2))

dropout
0    3003
1    1421
Name: count, dtype: int64
dropout
0    67.88
1    32.12
Name: proportion, dtype: float64


## 3. Feature Engineering

In [4]:
# Engineer Admission Grade (%)
df['admission_grade_pct'] = (df['Admission grade'] / 200.0) * 100.0

# Engineer Previous Semester Average Grade (%)
df['previous_semester_avg_grade_pct'] = (df['Curricular units 1st sem (grade)'] / 20.0) * 100.0

# Engineer Previous Semester Pass Ratio (%)
df['previous_semester_pass_ratio_pct'] = np.where(
    df['Curricular units 1st sem (enrolled)'] > 0,
    (df['Curricular units 1st sem (approved)'] / df['Curricular units 1st sem (enrolled)']) * 100.0,
    0.0
)

## 4. Column Renaming and Dropping Unused Columns

In [5]:
rename_map = {
    'Age at enrollment': 'age_at_enrollment',
    'Gender': 'gender',
    'Scholarship holder': 'scholarship_holder',
    'Debtor': 'debtor',
    'Displaced': 'displaced'
}
df = df.rename(columns=rename_map)

FINAL_COLUMNS = [
    'age_at_enrollment', 'gender', 'admission_grade_pct', 'scholarship_holder',
    'debtor', 'displaced', 'previous_semester_pass_ratio_pct',
    'previous_semester_avg_grade_pct', 'dropout'
]

df = df.loc[:, FINAL_COLUMNS]
assert list(df.columns) == FINAL_COLUMNS
assert df.shape[1] == 9
print("Cleaned dataframe shape:", df.shape)
df.head()

Cleaned dataframe shape: (4424, 9)


,age_at_enrollment,gender,admission_grade_pct,scholarship_holder,debtor,displaced,previous_semester_pass_ratio_pct,previous_semester_avg_grade_pct,dropout
0,20,1,63.65,0,0,1,0.000000,0.000000,1
1,19,1,71.25,0,0,1,100.000000,70.000000,0
2,19,1,62.40,0,0,1,0.000000,0.000000,1
3,20,0,59.80,0,0,1,100.000000,67.142857,0
4,45,0,70.75,0,0,0,83.333333,61.666667,0


## 5. Save Clean Dataset and Metadata

In [6]:
df.to_csv('../data/processed/student_dropout_8_features.csv', index=False)

metadata = {
    "features": [
        {"name": "age_at_enrollment", "type": "numeric"},
        {"name": "gender", "type": "binary"},
        {"name": "admission_grade_pct", "type": "numeric"},
        {"name": "scholarship_holder", "type": "binary"},
        {"name": "debtor", "type": "binary"},
        {"name": "displaced", "type": "binary"},
        {"name": "previous_semester_pass_ratio_pct", "type": "numeric"},
        {"name": "previous_semester_avg_grade_pct", "type": "numeric"}
    ],
    "target": "dropout"
}
with open('../artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

## 6. Train/Test Split

In [7]:
X = df.drop(columns=['dropout'])
y = df['dropout']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Train set size: 3539
Test set size: 885


## 7. Baseline Preprocessing and Modeling

In [8]:
continuous_cols = [
    'age_at_enrollment', 'admission_grade_pct', 
    'previous_semester_pass_ratio_pct', 'previous_semester_avg_grade_pct'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_cols)
    ],
    remainder='passthrough'
)

# Baseline models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }

with open('../reports/baseline_metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

print(json.dumps(results, indent=2))

{
  "Logistic Regression": {
    "accuracy": 0.8305084745762712,
    "precision": 0.7991071428571429,
    "recall": 0.6302816901408451,
    "f1": 0.7047244094488189,
    "roc_auc": 0.8828947060064212
  },
  "Decision Tree": {
    "accuracy": 0.7638418079096045,
    "precision": 0.629757785467128,
    "recall": 0.6408450704225352,
    "f1": 0.6352530541012217,
    "roc_auc": 0.7357455883386843
  },
  "Random Forest": {
    "accuracy": 0.8395480225988701,
    "precision": 0.8008474576271186,
    "recall": 0.6654929577464789,
    "f1": 0.7269230769230769,
    "roc_auc": 0.8727414403224673
  }
}
